In [1]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [2]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
print(train_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [5]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train_df["label"] = train_df["answer"].map(label_map)

In [6]:
def build_text(row):
    return (
        f"Question: {row['prompt']} "
        f"[SEP] A: {row['A']} "
        f"[SEP] B: {row['B']} "
        f"[SEP] C: {row['C']} "
        f"[SEP] D: {row['D']} "
        f"[SEP] E: {row['E']}"
    )

train_df["text"] = train_df.apply(build_text, axis=1)
test_df["text"] = test_df.apply(build_text, axis=1)

In [7]:
train_part, val_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

In [8]:
train_ds = Dataset.from_pandas(
    train_part[["text","label"]]
)

val_ds = Dataset.from_pandas(
    val_part[["text","label"]]
)

In [9]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds)
    }

In [12]:
training_args = TrainingArguments(
    output_dir="./bert_mcq",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.039510,0.745000
2,No log,0.519756,0.960000
3,No log,0.207815,0.990000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=339, training_loss=1.4478848887755809, metrics={'train_runtime': 162.1961, 'train_samples_per_second': 33.293, 'train_steps_per_second': 2.09, 'total_flos': 710418984652800.0, 'train_loss': 1.4478848887755809, 'epoch': 3.0})

In [15]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.20781490206718445,
 'eval_accuracy': 0.99,
 'eval_runtime': 2.0913,
 'eval_samples_per_second': 95.637,
 'eval_steps_per_second': 6.216,
 'epoch': 3.0}

In [16]:
from datasets import Dataset

test_ds = Dataset.from_pandas(
    test_df[["text"]]
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [17]:
predictions = trainer.predict(test_ds)

logits = predictions.predictions

probs = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()

In [18]:
labels = np.array(["A","B","C","D","E"])

top3 = []

for row in probs:
    idx = np.argsort(row)[::-1][:3]
    top3.append(" ".join(labels[idx]))

In [19]:
submission = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

submission["Prediction"] = top3   # note capital P

submission = submission[["ID", "Prediction"]]

submission.head()

,ID,Prediction
0,1,A C E
1,2,B D A
2,3,B D A
3,4,E A D
4,5,C A D


In [20]:
submission.to_csv(
    "submission_bert.csv",
    index=False
)

In [21]:
import os

print(os.listdir("/kaggle/working"))

['submission_bert.csv', 'bert_mcq', '__notebook__.ipynb']


In [22]:
import os

for f in os.listdir('/kaggle/working'):
    print(f)

submission_bert.csv
bert_mcq
__notebook__.ipynb


In [23]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

In [24]:
import os
print(os.listdir('/kaggle/working'))

['submission_bert.csv', 'bert_mcq', 'submission.csv', '__notebook__.ipynb']


In [25]:
df = pd.read_csv("submission.csv")
print(df.head())
print(df.columns.tolist())

   ID Prediction
0   1      A C E
1   2      B D A
2   3      B D A
3   4      E A D
4   5      C A D
['ID', 'Prediction']


In [1]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [2]:
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(torch.__version__)

2.10.0+cu128


In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

label2id = {"A":0,"B":1,"C":2,"D":3,"E":4}
id2label = {v:k for k,v in label2id.items()}

train_df["label"] = train_df["answer"].map(label2id)

print(train_df.shape)
print(test_df.shape)

(2000, 9)
(500, 7)


In [4]:
import re

def clean_prompt(text):
    text = str(text)

    patterns = [
        r"Pick the best possible answer.*?:",
        r"Choose the most appropriate answer.*?:",
        r"Select the correct answer.*?:",
        r"Among the following.*?:",
        r"Which of the following.*?:"
    ]

    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    return text.strip()

train_df["prompt"] = train_df["prompt"].apply(clean_prompt)
test_df["prompt"] = test_df["prompt"].apply(clean_prompt)

In [5]:
def make_text(df):
    return (
        df["prompt"] +
        " [SEP] " + df["A"] +
        " [SEP] " + df["B"] +
        " [SEP] " + df["C"] +
        " [SEP] " + df["D"] +
        " [SEP] " + df["E"]
    )

train_df["text"] = make_text(train_df)
test_df["text"] = make_text(test_df)

In [6]:
MODEL_NAME = "allenai/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [7]:
MAX_LENGTH = 384

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [8]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]],
    preserve_index=False
)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [10]:
training_args = TrainingArguments(
    output_dir="./scibert_mcq",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=1.5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    seed=42
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [12]:
trainer.train()

Step,Training Loss
100,2.957220
200,1.102551
300,0.072710
400,0.008408
500,0.005697


TrainOutput(global_step=500, training_loss=0.8293172732591629, metrics={'train_runtime': 334.4768, 'train_samples_per_second': 23.918, 'train_steps_per_second': 1.495, 'total_flos': 1578708854784000.0, 'train_loss': 0.8293172732591629, 'epoch': 4.0})

In [13]:
lengths = [
    len(tokenizer.encode(t, add_special_tokens=True))
    for t in train_df["text"][:200]
]

print("Max:", max(lengths))
print("Average:", sum(lengths) / len(lengths))

Max: 622
Average: 201.715


In [14]:
test_logits = trainer.predict(test_dataset).predictions

print(test_logits.shape)

(500, 5)


In [15]:
import torch

probs = torch.softmax(
    torch.tensor(test_logits),
    dim=1
).numpy()

print(probs.shape)
print(probs[0])
print(probs[0].sum())

(500, 5)
[9.9809963e-01 3.4085065e-04 3.7068268e-04 6.5003621e-04 5.3886761e-04]
1.0000001


In [16]:
labels = np.array(["A", "B", "C", "D", "E"])

predictions = []

for p in probs:
    top3 = np.argsort(p)[::-1][:3]
    predictions.append(" ".join(labels[top3]))

In [17]:
submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": predictions
})

submission.head()

,ID,Prediction
0,1,A D E
1,2,B C A
2,3,B A E
3,4,E C D
4,5,C A D


In [18]:
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(submission.shape)

   ID Prediction
0   1      A D E
1   2      B C A
2   3      B A E
3   4      E C D
4   5      C A D
(500, 2)


In [19]:
submission.to_csv("submission.csv", index=False)

In [20]:
import os
print(os.listdir("/kaggle/working"))


['__notebook__.ipynb', 'submission.csv', 'scibert_mcq']


In [21]:
submission.head()

,ID,Prediction
0,1,A D E
1,2,B C A
2,3,B A E
3,4,E C D
4,5,C A D


In [22]:
import os

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'submission.csv', 'scibert_mcq']


In [23]:
import os

if os.path.exists("/kaggle/working/submission.csv"):
    os.remove("/kaggle/working/submission.csv")

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'submission.csv', 'scibert_mcq']


In [24]:
import os

submission.to_csv("submission.csv", index=False)

os.system("sync")

print(os.path.exists("/kaggle/working/submission.csv"))

True


In [25]:
submission.head()

,ID,Prediction
0,1,A D E
1,2,B C A
2,3,B A E
3,4,E C D
4,5,C A D


In [26]:
import os

print(os.getcwd())
print(os.listdir())
print(os.listdir("/kaggle/working"))

/kaggle/working
['__notebook__.ipynb', 'submission.csv', 'scibert_mcq']
['__notebook__.ipynb', 'submission.csv', 'scibert_mcq']


In [27]:
import pandas as pd

submission = pd.read_csv("/kaggle/working/submission.csv")

print(submission.head())
print(submission.shape)

   ID Prediction
0   1      A D E
1   2      B C A
2   3      B A E
3   4      E C D
4   5      C A D
(500, 2)


In [28]:
import os

print("Exists:", os.path.exists("/kaggle/working/submission.csv"))
print("Size:", os.path.getsize("/kaggle/working/submission.csv"))

Exists: True
Size: 4906


In [1]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [2]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
print(train_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [5]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train_df["label"] = train_df["answer"].map(label_map)

In [6]:
def build_text(row):
    return (
        f"Question: {row['prompt']} "
        f"[SEP] A: {row['A']} "
        f"[SEP] B: {row['B']} "
        f"[SEP] C: {row['C']} "
        f"[SEP] D: {row['D']} "
        f"[SEP] E: {row['E']}"
    )

train_df["text"] = train_df.apply(build_text, axis=1)
test_df["text"] = test_df.apply(build_text, axis=1)

In [7]:
train_part, val_part = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

In [8]:
train_ds = Dataset.from_pandas(
    train_part[["text","label"]]
)

val_ds = Dataset.from_pandas(
    val_part[["text","label"]]
)

In [9]:
MODEL_NAME = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds)
    }

In [11]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [12]:
training_args = TrainingArguments(
    output_dir="./distilroberta_mcq",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    warmup_steps=100,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.415679,0.610000
2,No log,0.075398,0.995000
3,No log,0.004272,1.000000
4,No log,0.003139,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=452, training_loss=1.0224860537368639, metrics={'train_runtime': 128.1685, 'train_samples_per_second': 56.176, 'train_steps_per_second': 3.527, 'total_flos': 476908148736000.0, 'train_loss': 1.0224860537368639, 'epoch': 4.0})

In [15]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.0031389968935400248,
 'eval_accuracy': 1.0,
 'eval_runtime': 1.0768,
 'eval_samples_per_second': 185.728,
 'eval_steps_per_second': 12.072,
 'epoch': 4.0}

In [16]:
from datasets import Dataset

test_ds = Dataset.from_pandas(
    test_df[["text"]]
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [17]:
predictions = trainer.predict(test_ds)

logits = predictions.predictions

probs = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()

In [18]:
labels = np.array(["A","B","C","D","E"])

top3 = []

for row in probs:
    idx = np.argsort(row)[::-1][:3]
    top3.append(" ".join(labels[idx]))

In [19]:
submission = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

submission["Prediction"] = top3   # note capital P

submission = submission[["ID", "Prediction"]]

submission.head()

,ID,Prediction
0,1,A E D
1,2,B C E
2,3,B C D
3,4,E D A
4,5,C B A


In [20]:
submission.to_csv(
    "submission_distilroberta.csv",
    index=False
)

In [21]:
import os

print(os.listdir("/kaggle/working"))

['__notebook__.ipynb', 'submission_distilroberta.csv', 'distilroberta_mcq']


In [22]:
import os

for f in os.listdir('/kaggle/working'):
    print(f)

__notebook__.ipynb
submission_distilroberta.csv
distilroberta_mcq


In [23]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

In [24]:
import os
print(os.listdir('/kaggle/working'))

['__notebook__.ipynb', 'submission.csv', 'submission_distilroberta.csv', 'distilroberta_mcq']


In [25]:
df = pd.read_csv("submission.csv")
print(df.head())
print(df.columns.tolist())

   ID Prediction
0   1      A E D
1   2      B C E
2   3      B C D
3   4      E D A
4   5      C B A
['ID', 'Prediction']


In [1]:
!pip -q install transformers datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [2]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
def prepare_text(df):

    df = df.copy()

    df["text"] = (
        "Question:\n"
        + df["prompt"].fillna("")
        + "\n\nChoices:\n"
        + "A. " + df["A"].fillna("") + "\n"
        + "B. " + df["B"].fillna("") + "\n"
        + "C. " + df["C"].fillna("") + "\n"
        + "D. " + df["D"].fillna("") + "\n"
        + "E. " + df["E"].fillna("") + "\n\n"
        + "Select the correct answer."
    )

    return df


train_df = prepare_text(train_df)
test_df = prepare_text(test_df)

train_df["label"] = train_df["answer"].map({
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
})

In [5]:
MODEL_NAME = "google/electra-base-discriminator"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

print("Loaded Successfully!")

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

Loaded Successfully!


In [6]:
train_data, valid_data = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"]
)

In [7]:
train_dataset = Dataset.from_pandas(
    train_data[["text", "label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_data[["text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]]
)

In [8]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=384
    )

In [9]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

valid_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask"
    ]
)

In [11]:
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = (predictions == labels).mean()

    return {
        "accuracy": accuracy
    }

In [12]:
training_args = TrainingArguments(

    output_dir="./electra_mcq",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=5,

    warmup_steps=100,

    weight_decay=0.01,

    load_best_model_at_end=True,

    report_to="none"
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.720300,0.625000
2,No log,0.409945,1.000000
3,No log,0.028738,1.000000
4,No log,0.012488,1.000000
5,1.099137,0.010277,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=565, training_loss=0.974666050050111, metrics={'train_runtime': 415.1761, 'train_samples_per_second': 21.678, 'train_steps_per_second': 1.361, 'total_flos': 1776047461632000.0, 'train_loss': 0.974666050050111, 'epoch': 5.0})

In [15]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.01027738582342863,
 'eval_accuracy': 1.0,
 'eval_runtime': 2.9904,
 'eval_samples_per_second': 66.882,
 'eval_steps_per_second': 4.347,
 'epoch': 5.0}

In [16]:
predictions = trainer.predict(test_dataset)

logits = predictions.predictions

In [17]:
import numpy as np
import torch

probs = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()

labels = np.array(["A", "B", "C", "D", "E"])

top3 = np.argsort(
    probs,
    axis=1
)[:, -3:][:, ::-1]

predictions = [
    " ".join(labels[idx])
    for idx in top3
]

In [18]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,prediction
0,1,A D E
1,2,B C E
2,3,B C E
3,4,E D C
4,5,C D A


In [19]:
import os
import shutil
import pandas as pd

assert os.path.exists("submission.csv"), "submission.csv not found!"

submission = pd.read_csv("submission.csv")
print(submission.head())

os.makedirs("/kaggle/working/final_submission", exist_ok=True)
shutil.copy("submission.csv", "/kaggle/working/final_submission/submission.csv")

print("Saved successfully!")

   id prediction
0   1      A D E
1   2      B C E
2   3      B C E
3   4      E D C
4   5      C D A
Saved successfully!


In [20]:
import os

print(os.listdir("/kaggle/working"))
print(os.listdir("/kaggle/working/final_submission"))

['electra_mcq', '__notebook__.ipynb', 'submission.csv', 'final_submission']
['submission.csv']
